# Experiment 4: Regular Expressions for Text Processing

**Mapped Course Outcome:** CO3  
**BT Level Mapping:** BT3

## Aim

To use Python regular expressions for searching, extracting, cleaning, validating, and transforming structured patterns in text.

## Objectives

- Understand regex syntax and pattern matching.
- Extract emails, phone numbers, dates, hashtags, and repeated whitespace.
- Use `search`, `findall`, `sub`, and `split` operations.
- Build reusable patterns for text preprocessing tasks.

## Tools / Requirements

- Python 3.10 or above
- Any Python IDE or notebook
- Python standard-library module: `re`

## Theory

A regular expression (regex) is a compact pattern language used to describe sets of strings. In NLP preprocessing, regex can extract structured tokens, clean noisy text, validate fields, split text, and replace patterns. Common constructs include character classes (`[A-Za-z]`), quantifiers (`+`, `*`, `{m,n}`), anchors, groups, alternation (`|`), and look-around assertions.

Python supplies regex operations through the built-in `re` module. Raw strings (for example, `r"\\d+"`) are preferred because they avoid conflicts between Python escaping and regex escaping. Regex is effective for shallow pattern matching, but it cannot replace syntactic or semantic analysis where context and ambiguity are important.

## Methodology

1. Create a synthetic text sample containing emails, phone numbers, dates, hashtags, and irregular spacing.
2. Define a regex pattern for each target structure.
3. Use `re.findall()` to extract every matching item.
4. Use `re.search()`, `re.sub()`, and `re.split()` for searching, normalization, masking, and sentence splitting.
5. Compile patterns for repeated processing, then inspect matches and limitations.

## Dataset

The synthetic paragraph below represents customer messages, web text, logs, social-media posts, or domain documents that may contain structured patterns.

In [1]:
import re

sample_text = """
Hello   team! Contact Alice at alice.johnson@example.com or support+nlp@university.edu.
Call +91 98765 43210, 987-654-3210, or (555) 123-4567 for help.
The workshops are on 14-08-2026, 2026/08/15, and 16.08.2026.
Follow #NLP, #TextProcessing, and #Python3 for updates.
This     sentence contains irregular     spacing.
"""

print(sample_text)


Hello   team! Contact Alice at alice.johnson@example.com or support+nlp@university.edu.
Call +91 98765 43210, 987-654-3210, or (555) 123-4567 for help.
The workshops are on 14-08-2026, 2026/08/15, and 16.08.2026.
Follow #NLP, #TextProcessing, and #Python3 for updates.
This     sentence contains irregular     spacing.



In [4]:
# Raw strings keep regex escaping readable.
EMAIL_PATTERN = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
PHONE_PATTERN = (
    r"(?<!\w)(?:(?:\+91[-\s]?)?\d{5}[-\s]?\d{5}|"
    r"(?:\(\d{3}\)|\d{3})[-\s]\d{3}[-\s]\d{4})(?!\w)"
)
DATE_PATTERN = r"(?<!\d)(?:\d{2}[-.]\d{2}[-.]\d{4}|\d{4}/\d{2}/\d{2})(?!\d)"
HASHTAG_PATTERN = r"(?<!\w)#\w+"
WHITESPACE_PATTERN = r"\s+"

patterns = {
    "emails": re.compile(EMAIL_PATTERN),
    "phone_numbers": re.compile(PHONE_PATTERN),
    "dates": re.compile(DATE_PATTERN),
    "hashtags": re.compile(HASHTAG_PATTERN),
}

patterns

{'emails': re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
            re.UNICODE),
 'phone_numbers': re.compile(r'(?<!\w)(?:(?:\+91[-\s]?)?\d{5}[-\s]?\d{5}|(?:\(\d{3}\)|\d{3})[-\s]\d{3}[-\s]\d{4})(?!\w)',
            re.UNICODE),
 'dates': re.compile(r'(?<!\d)(?:\d{2}[-.]\d{2}[-.]\d{4}|\d{4}/\d{2}/\d{2})(?!\d)',
            re.UNICODE),
 'hashtags': re.compile(r'(?<!\w)#\w+', re.UNICODE)}

In [5]:
# findall() returns all non-overlapping matches in the text.
extracted = {name: pattern.findall(sample_text) for name, pattern in patterns.items()}

for entity, matches in extracted.items():
    print(f"{entity.replace('_', ' ').title()}: {matches}")

Emails: ['alice.johnson@example.com', 'support+nlp@university.edu']
Phone Numbers: ['+91 98765 43210', '987-654-3210', '(555) 123-4567']
Dates: ['14-08-2026', '2026/08/15', '16.08.2026']
Hashtags: ['#NLP', '#TextProcessing', '#Python3']


In [6]:
# search() returns the first match object (or None when no match is present).
first_email = patterns["emails"].search(sample_text)
if first_email:
    print("First email:", first_email.group())
    print("Location (start, end):", first_email.span())

# A fullmatch() is suitable when a whole field must follow the expected format.
email_candidates = ["learner_01@college.edu", "invalid-email@", "admin@example.org"]
for candidate in email_candidates:
    is_valid = bool(patterns["emails"].fullmatch(candidate))
    print(f"{candidate:24} -> valid: {is_valid}")

First email: alice.johnson@example.com
Location (start, end): (32, 57)
learner_01@college.edu   -> valid: True
invalid-email@           -> valid: False
admin@example.org        -> valid: True


In [7]:
# sub() replaces every repeated whitespace sequence with one space.
normalized_text = re.sub(WHITESPACE_PATTERN, " ", sample_text).strip()
print("Normalized text:\n", normalized_text)

# A replacement function can mask personal information while retaining the domain.
def mask_email(match):
    local_part, domain = match.group().split("@", maxsplit=1)
    return f"{local_part[0]}***@{domain}"

masked_text = patterns["emails"].sub(mask_email, normalized_text)
print("\nText with masked emails:\n", masked_text)

Normalized text:
 Hello team! Contact Alice at alice.johnson@example.com or support+nlp@university.edu. Call +91 98765 43210, 987-654-3210, or (555) 123-4567 for help. The workshops are on 14-08-2026, 2026/08/15, and 16.08.2026. Follow #NLP, #TextProcessing, and #Python3 for updates. This sentence contains irregular spacing.

Text with masked emails:
 Hello team! Contact Alice at a***@example.com or s***@university.edu. Call +91 98765 43210, 987-654-3210, or (555) 123-4567 for help. The workshops are on 14-08-2026, 2026/08/15, and 16.08.2026. Follow #NLP, #TextProcessing, and #Python3 for updates. This sentence contains irregular spacing.


In [8]:
# split() separates sentences only when punctuation is followed by whitespace and a new sentence starter.
sentences = [
    sentence.strip()
    for sentence in re.split(r"(?<=[.!?])\s+(?=[A-Z#])", normalized_text)
    if sentence.strip()
]

for number, sentence in enumerate(sentences, start=1):
    print(f"{number}. {sentence}")

# Reusing compiled patterns avoids recompiling each expression in repeated processing.
messages = [
    "Email research.lab@college.edu about #NLP.",
    "No contact information is available.",
    "Call 98765 43210 before 14-08-2026.",
]

for message in messages:
    summary = {name: pattern.findall(message) for name, pattern in patterns.items()}
    print(f"\nMessage: {message}")
    print(summary)

1. Hello team!
2. Contact Alice at alice.johnson@example.com or support+nlp@university.edu.
3. Call +91 98765 43210, 987-654-3210, or (555) 123-4567 for help.
4. The workshops are on 14-08-2026, 2026/08/15, and 16.08.2026.
5. Follow #NLP, #TextProcessing, and #Python3 for updates.
6. This sentence contains irregular spacing.

Message: Email research.lab@college.edu about #NLP.
{'emails': ['research.lab@college.edu'], 'phone_numbers': [], 'dates': [], 'hashtags': ['#NLP']}

Message: No contact information is available.
{'emails': [], 'phone_numbers': [], 'dates': [], 'hashtags': []}

Message: Call 98765 43210 before 14-08-2026.
{'emails': [], 'phone_numbers': ['98765 43210'], 'dates': ['14-08-2026'], 'hashtags': []}


## Results

The program extracts emails, phone numbers, dates, and hashtags into separate lists. It also identifies the first email and its position, validates complete email-field candidates, normalizes repeated whitespace, masks email addresses, and splits the paragraph into sentences. Compiled patterns are reused to process multiple messages.

## Discussion and Limitations

The patterns are intentionally suitable for common examples, not every real-world format. For example, the phone pattern supports the Indian and US-style formats used in the sample, but not all country codes or extensions. The email pattern does not verify whether an address actually exists, and the date pattern checks shape rather than calendar validity (for example, it may match an impossible day). Production systems should adapt patterns to the domain, add validation where required, and test both false positives and missed cases.

## Outcomes

- Applied regex patterns for common text entities.
- Used extraction, search, substitution, splitting, and validation functions correctly.
- Recognized limitations and overmatching risks in regex design.
- Applied regex as a preprocessing step in a larger NLP pipeline.

## Resources

- [Python `re` documentation](https://docs.python.org/3/library/re.html)
- [NLTK Book, Chapter 3](https://www.nltk.org/book/ch03.html)
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search](https://www.youtube.com/results?search_query=Python+regular+expressions+for+NLP+text+processing)